In [ ]:
# Parameters (will be overridden by URL query params when running with Voila)
entityId = "default_entity_id"
# Backend base URL comes from environment when running under Voila
# Set CASE_MGMT_BACKEND_URL in the environment; this default is for local dev only.
import os
backendUrl = os.getenv("CASE_MGMT_BACKEND_URL", "http://localhost:3000")
# Shared secret for proxy requests (set JUPYTER_SHARED_SECRET in environment)
JUPYTER_SHARED_SECRET = os.getenv('JUPYTER_SHARED_SECRET')
if JUPYTER_SHARED_SECRET:
    headers = {'x-jupyter-secret': JUPYTER_SHARED_SECRET}
else:
    headers = {}
filter = "Last Month"
# Fallback account ID for Benford's Law analysis
benfordAccountId = "cdtrAcct_3a1f3d24fb2046f2a28dc1fa506d6d69"


In [ ]:
try:
    import requests
    import pandas as pd
    import plotly.graph_objects as go
    import plotly.express as px
    import plotly.io as pio
    from plotly.subplots import make_subplots
    from IPython.display import display, HTML
    import os
    
    # Set default renderer to basic html/notebook
    pio.renderers.default = "notebook"

    # Output styling
    display(HTML("""
    <style>
        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
        .metric-card { background: white; padding: 15px; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); text-align: center; }
        .metric-value { font-size: 24px; font-weight: bold; color: #111827; }
        .metric-label { font-size: 12px; color: #6B7280; text-transform: uppercase; letter-spacing: 0.05em; }
        .grid-container { display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin-bottom: 20px; }
        .table-container { margin-top: 20px; overflow-x: auto; }
        table { width: 100%; border-collapse: collapse; font-size: 14px; }
        th { text-align: left; padding: 12px; border-bottom: 1px solid #e5e7eb; color: #6B7280; font-weight: 600; }
        td { padding: 12px; border-bottom: 1px solid #e5e7eb; color: #111827; }
        tr:last-child td { border-bottom: none; }
    </style>
    """))
except Exception as e:
    print(f"Import Error: {e}")

In [ ]:
# Fetch Data Helper
def fetch_data(entity_id, backend_url, time_filter):
    url = f"{backend_url}/api/v1/jupyter/proxy/transaction-history/{entity_id}"
    params = {'tenantId': 'DEFAULT'}
    if time_filter:
        params['filter'] = time_filter
    
    try:
        response = requests.get(url, params=params, headers=headers)
        # Provide clearer errors instead of raising deep exceptions to Voila
        if not response.ok:
            try:
                print(f"Request to {url} failed: {response.status_code} {response.reason}")
                print('Response body:', response.text)
            except Exception:
                pass
            return None
        return response.json()
    except Exception as e:
        import traceback
        print(f"Exception during fetch_data: {e}")
        traceback.print_exc()
        return None

# Main Fetch Logic with Fallback
# Ensure variables exist even when fetch fails
data = None
fallback_id = "cdtrAcct_3a1f3d24fb2046f2a28dc1fa506d6d69"
summary = {}
timeline = []
volume_dist = []
cumulative = []
recent = []
df_timeline = pd.DataFrame()
df_volume = pd.DataFrame()
df_cumulative = pd.DataFrame()
df_recent = pd.DataFrame()

try:
    # 1. Try requested entityId
    if entityId and entityId != "default_entity_id":
        data = fetch_data(entityId, backendUrl, filter)

    # 2. If Failed/Empty or Default, try fallback (Dev environment only)
    if not data:
        data = fetch_data(fallback_id, backendUrl, filter)
        
    if data:
        summary = data.get('summary', {})
        timeline = data.get('timeline', [])
        volume_dist = data.get('volumeDistribution', [])
        # Accept alternative keys
        if not volume_dist:
            volume_dist = data.get('volume_dist') or data.get('volume') or []
        cumulative = data.get('cumulative', [])
        recent = data.get('recentTransactions', [])
        
        df_timeline = pd.DataFrame(timeline)
        if not df_timeline.empty:
            df_timeline['date'] = pd.to_datetime(df_timeline['date'])

        # Normalise volume distribution from backend (bucketStart/transactionCount)
        df_volume = pd.DataFrame(volume_dist)

        if not df_volume.empty:
            if 'bucketStart' in df_volume.columns:
                df_volume['date'] = pd.to_datetime(df_volume['bucketStart'])
            elif 'date' in df_volume.columns:
                df_volume['date'] = pd.to_datetime(df_volume['date'])
            else:
                df_volume['date'] = pd.NaT

            # Ensure we have a transaction-count column for charts
            if 'transactionCount' in df_volume.columns:
                df_volume['txCount'] = df_volume['transactionCount']
            elif 'bucket_tx_count' in df_volume.columns:
                df_volume['txCount'] = df_volume['bucket_tx_count']
            elif 'txCount' in df_volume.columns:
                df_volume['txCount'] = df_volume['txCount']
            elif 'count' in df_volume.columns:
                df_volume['txCount'] = df_volume['count']
            else:
                df_volume['txCount'] = 0
            
        df_cumulative = pd.DataFrame(cumulative)
        if not df_cumulative.empty:
            df_cumulative['date'] = pd.to_datetime(df_cumulative['date'])
            
        df_recent = pd.DataFrame(recent)
            
except Exception as e:
    import traceback
    display(HTML(f"<div style='color: red; padding: 20px; border: 1px solid red; border-radius: 5px;'>Error fetching data: {str(e)}</div>"))
    print(f"Backend URL: {backendUrl}")
    traceback.print_exc()

In [ ]:
if data:
    # Display Metrics
    total_vol = f"${summary.get('totalVolume', 0):,.2f}"
    total_tx = f"{summary.get('totalTransactions', 0):,}"
    alerts = f"{summary.get('alertsTriggered', 0)}"
    investigated = f"{summary.get('investigated', 0)}"

    display(HTML(f"""
    <div class=\"grid-container\">
        <div class=\"metric-card\">
            <div class=\"metric-label\">Total Volume</div>
            <div class=\"metric-value\">{total_vol}</div>
        </div>
        <div class=\"metric-card\">
            <div class=\"metric-label\">Total Transactions</div>
            <div class=\"metric-value\">{total_tx}</div>
        </div>
        <div class=\"metric-card\">
            <div class=\"metric-label\">Alerts Triggered</div>
            <div class=\"metric-value\">{alerts}</div>
        </div>
        <div class=\"metric-card\">
            <div class=\"metric-label\">Investigated</div>
            <div class=\"metric-value\">{investigated}</div>
        </div>
    </div>
    """))

In [ ]:
if data and not df_timeline.empty:
    # Prepare daily aggregates for dual-axis timeline (amount + count)
    df_daily = None
    if 'date' in df_timeline.columns:
        tmp = df_timeline.copy()
        tmp['date_only'] = tmp['date'].dt.date
        df_daily = (
            tmp.groupby('date_only')
               .agg(txCount=('transactionId', 'count'))
               .reset_index()
        )
        df_daily['date'] = pd.to_datetime(df_daily['date_only'])
        df_daily = df_daily[['date', 'txCount']]

    # Create Charts: 3 rows -> Timeline (dual axis), Cumulative, Volume Distribution
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=(
            "Transaction Timeline (Amount & Count)",
            "Cumulative Value (Running Total)",
            # "Transaction Volume Distribution (Count)"
        ),
        row_heights=[0.4, 0.3, 0.3],
        specs=[
            [{"secondary_y": True}],  # Row 1: dual y-axis
            [{}],                      # Row 2
            [{}],                      # Row 3
        ],
    )

    # 1. Transaction Timeline (Amount on left Y-axis)
    fig.add_trace(
        go.Scatter(
            x=df_timeline['date'],
            y=df_timeline['amount'],
            mode='markers+lines',
            name='Amount',
            marker=dict(
                size=8,
                color=df_timeline['isAlerted'].map({True: 'red', False: 'blue'}),
                symbol=df_timeline['isAlerted'].map({True: 'circle', False: 'circle'}),
            ),
            hovertemplate="<b>Date:</b> %{x}<br><b>Amount:</b> $%{y:,.2f}<extra></extra>",
        ),
        row=1,
        col=1,
        secondary_y=False,
    )

    # 1b. Transaction Count (Right Y-axis)
    if df_daily is not None and not df_daily.empty:
        fig.add_trace(
            go.Scatter(
                x=df_daily['date'],
                y=df_daily['txCount'],
                mode='lines+markers',
                name='Transaction Count',
                line=dict(color='#F59E0B', width=2, dash='dot'),
                marker=dict(size=6, color='#F59E0B'),
                hovertemplate="<b>Date:</b> %{x}<br><b>Transactions:</b> %{y}<extra></extra>",
            ),
            row=1,
            col=1,
            secondary_y=True,
        )

    # 2. Cumulative Value (Running Total)
    if not df_cumulative.empty:
        fig.add_trace(
            go.Scatter(
                x=df_cumulative['date'],
                y=df_cumulative['cumulativeAmount'],
                fill='tozeroy',
                mode='lines',
                name='Cumulative Value',
                line=dict(color='#10B981'),  # Green
            ),
            row=2,
            col=1,
        )

    # 3. Transaction Volume Distribution (Bar chart of counts)
    if not df_volume.empty and 'txCount' in df_volume.columns:
        fig.add_trace(
            go.Bar(
                x=df_volume['date'],
                y=df_volume['txCount'],
                name='Transaction Count per Bucket',
                marker_color='#6366F1',
            ),
            row=3,
            col=1,
        )

    # Layout & axes styling
    fig.update_layout(
        height=800,
        showlegend=False,
        margin=dict(l=20, r=20, t=40, b=20),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
    )

    # Grid lines
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#e5e7eb')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#e5e7eb')

    # Axis labels for the dual-axis timeline
    fig.update_yaxes(title_text="Amount ($)", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Transaction Count", row=1, col=1, secondary_y=True)

    fig.show()

In [ ]:
# Volume Distribution: ensure a chart is shown (placeholder if no data)
try:
    import plotly.graph_objects as go
    if df_volume is None or df_volume.empty:
        display(HTML("<div style='color:#ef4444; padding:12px; border:1px solid #fee2e2; border-radius:6px;'>No volume distribution data available to plot.</div>"))
        fig = go.Figure()
        fig.update_layout(height=300, title="Transaction Volume Distribution (no data)")
        fig.show()
    else:
        if 'date' in df_volume.columns and 'txCount' in df_volume.columns:
            fig = go.Figure(go.Bar(x=df_volume['date'], y=df_volume['txCount'], marker_color='#6366F1'))
            fig.update_layout(height=300, title='Transaction Volume Distribution (Count)', xaxis_title='Date', yaxis_title='Tx Count')
            fig.show()
        else:
            display(HTML("<div style='color:#ef4444; padding:12px;'>Volume distribution present but missing expected columns ('date' and 'txCount').</div>"))
except Exception as e:
    import traceback
    print('Error rendering volume distribution:', e)
    traceback.print_exc()


In [ ]:
# Fetch Benford's Law Data and Create Analysis Chart
try:
    from datetime import datetime, timedelta
    end_date = datetime.now()
    start_date = end_date - timedelta(days=90)

    target_account_id = entityId if entityId and entityId != "default_entity_id" else benfordAccountId

    benford_url = f"{backendUrl}/api/v1/jupyter/proxy/lake/analytics/benford/account/{target_account_id}"
    benford_params = {
        'tenantId': 'DEFAULT',
        'from': start_date.strftime('%Y-%m-%d'),
        'to': end_date.strftime('%Y-%m-%d')
    }

    try:
        benford_response = requests.get(benford_url, params=benford_params, headers=headers, timeout=20)
        if benford_response.status_code == 404 and target_account_id != benfordAccountId:
            print(f"Benford data not found for {target_account_id}, trying fallback: {benfordAccountId}")
            benford_url = f"{backendUrl}/api/v1/jupyter/proxy/lake/analytics/benford/account/{benfordAccountId}"
            benford_response = requests.get(benford_url, params=benford_params, headers=headers, timeout=20)

        if not benford_response.ok:
            print(f"Benford request failed: {benford_response.status_code} {benford_response.reason}")
            try:
                print('Response body:', benford_response.text)
            except Exception:
                pass
            benford_data = None
        else:
            benford_response.raise_for_status()
            benford_data = benford_response.json()
            # Debug: print raw benford response summary
            # try:
            #     print('Benford response top-level keys:', list(benford_data.keys()) if isinstance(benford_data, dict) else type(benford_data))
            #     if isinstance(benford_data, dict):
            #         print('sample expected (first 3):', dict(list(benford_data.get('expected', {}).items())[:3]))
            #         print('sample actual (first 3):', dict(list(benford_data.get('actual', {}).items())[:3]))
            #         print('sampleSize:', benford_data.get('sampleSize'))
            # except Exception:
            #     pass

    except Exception as e:
        import traceback
        print("Exception during Benford fetch:", e)
        traceback.print_exc()
        benford_data = None

    # Process benford_data if present
    expected_benford = []
    actual_benford = []
    digits = [1,2,3,4,5,6,7,8,9]

    if benford_data and isinstance(benford_data, dict) and 'expected' in benford_data and 'actual' in benford_data:
        exp_dict = benford_data['expected']
        act_dict = benford_data['actual']
        sample_size = float(benford_data.get('sampleSize', 0) or 0)
        for d in digits:
            d_str = str(d)
            expected_benford.append(float(exp_dict.get(d_str, 0)) * 100)
            count = float(act_dict.get(d_str, 0))
            actual_benford.append((count / sample_size * 100) if sample_size > 0 else 0)
    else:
        if benford_data is None:
            print('No Benford data returned from backend')
        else:
            print('Benford response format unexpected:', type(benford_data))

    # Ensure arrays of length 9 for plotting
    if len(expected_benford) != 9:
        expected_benford = [30.1, 17.6, 12.5, 9.7, 7.9, 6.7, 5.8, 5.1, 4.6]
    if len(actual_benford) != 9:
        actual_benford = [0] * 9

except Exception as e:
    import traceback
    print(f"Benford analysis error: {e}")
    traceback.print_exc()


In [ ]:
# Benford chart and table (always render; fallbacks applied earlier)
try:
    import plotly.graph_objects as go
    import pandas as pd

    digits = [1,2,3,4,5,6,7,8,9]
    # Ensure expected_benford and actual_benford exist
    try:
        exp = expected_benford
        act = actual_benford
    except NameError:
        exp = [30.1, 17.6, 12.5, 9.7, 7.9, 6.7, 5.8, 5.1, 4.6]
        act = [0]*9

    fig = go.Figure()
    fig.add_trace(go.Bar(x=digits, y=exp, name='Expected (%)', marker_color='#10B981'))
    fig.add_trace(go.Bar(x=digits, y=act, name='Actual (%)', marker_color='#6366F1'))
    fig.update_layout(barmode='group', title='Benford Analysis', xaxis_title='Leading Digit', yaxis_title='Percentage', height=420)
    fig.show()

    # Show a small summary table
    try:
        df_b = pd.DataFrame({'digit': digits, 'expected_pct': exp, 'actual_pct': act})
        display(df_b.style.hide_index())
    except Exception:
        pass
except Exception as e:
    import traceback
    print('Error rendering Benford chart/table:', e)
    traceback.print_exc()


In [ ]:
if data and not df_recent.empty:
    display(HTML("<h3>Recent Transactions</h3>"))

    # Normalise status for display, but keep the real values
    if 'status' in df_recent.columns:
        def _format_status(v):
            # Backend sends this as a list of labels like ["Alert", "Investigated"]
            if isinstance(v, (list, tuple)):
                return ", ".join(str(x) for x in v)
            if v is None or (isinstance(v, float) and pd.isna(v)):
                return ""
            return str(v)

        df_recent['status'] = df_recent['status'].apply(_format_status)

    # Select and rename columns for display
    cols = ['date', 'type', 'counterparty', 'amount', 'currency', 'status']
    # Ensure cols exist
    valid_cols = [c for c in cols if c in df_recent.columns]
    display_df = df_recent[valid_cols].copy()
    
    # Format Amount
    if 'amount' in display_df.columns:
        display_df['amount'] = display_df['amount'].apply(lambda x: f"{x:,.2f}")
        
    # Render HTML table with styling
    html = display_df.to_html(index=False, classes='table')
    display(HTML(f"<div class='table-container'>{html}</div>"))